[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pinecone-io/examples/blob/master/learn/generation/langchain/handbook/10-langchain-multi-query.ipynb) [![Open nbviewer](https://raw.githubusercontent.com/pinecone-io/examples/master/assets/nbviewer-shield.svg)](https://nbviewer.org/github/pinecone-io/examples/blob/master/learn/generation/langchain/handbook/08-langchain-multi-query.ipynb)

#### [LangChain Handbook](https://pinecone.io/learn/langchain)

# LangChain Multi-Query for RAG

In [1]:
# Use %pip (not !pip) so packages install into THIS notebook's kernel,
# not some other Python on your system. Restart the kernel after this runs.
%pip install -qU   langchain-qdrant==0.2.1 qdrant-client   langchain==0.3.25   langchain-community==0.3.25   langchain-google-genai==2.0.10   sniffio anyio pyparsing   datasets   tiktoken

# --- OpenRouter alternative (for reference) ---
# %pip install -qU langchain-openai==0.3.22

Note: you may need to restart the kernel to use updated packages.


## Getting Data

We will download an existing dataset from Hugging Face Datasets.

In [2]:
from datasets import load_dataset

data = load_dataset("jamescalam/ai-arxiv-chunked", split="train")
data

Dataset({
    features: ['doi', 'chunk-id', 'chunk', 'id', 'title', 'summary', 'source', 'authors', 'categories', 'comment', 'journal_ref', 'primary_category', 'published', 'updated', 'references'],
    num_rows: 41584
})

In [3]:
from langchain.docstore.document import Document

docs = []

for row in data:
    doc = Document(
        page_content=row["chunk"],
        metadata={
            "title": row["title"],
            "source": row["source"],
            "id": row["id"],
            "chunk-id": row["chunk-id"],
            "text": row["chunk"]
        }
    )
    docs.append(doc)

## Embedding and Vector DB Setup

Initialize our embedding model:

In [4]:
import os
from getpass import getpass
from langchain_google_genai import GoogleGenerativeAIEmbeddings

# get Google API key from https://aistudio.google.com/app/apikey
os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY") or getpass("Enter your Google API key: ")
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")

# Qdrant needs no API key for this deployment - we only point at its URL.
QDRANT_URL = "http://109.199.118.73:6333"

# --- OpenRouter alternative (for reference) ---
# os.environ["OPENROUTER_API_KEY"] = os.getenv("OPENROUTER_API_KEY") or getpass("Enter your OpenRouter API key: ")
# OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")

embed = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001",
    google_api_key=GOOGLE_API_KEY,
)

# --- OpenAI backup (for reference) ---
# from langchain.embeddings.openai import OpenAIEmbeddings
# OPENAI_API_KEY = os.getenv('OPENAI_API_KEY') or getpass("OpenAI API Key: ")
# embed = OpenAIEmbeddings(
#     model="text-embedding-ada-002", openai_api_key=OPENAI_API_KEY, disallowed_special=()
# )

D:\Work\DEBI\Agents\.venv-langchain\Lib\site-packages\langchain_google_genai\chat_models.py:47: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  from google.generativeai.caching import CachedContent  # type: ignore[import]


Now we create our vector DB to store our vectors. For this we use [Qdrant](https://qdrant.tech/), pointing at an existing Qdrant deployment. This deployment requires no API key, so we only need its URL.

In [5]:
from qdrant_client import QdrantClient

# connect to the existing Qdrant deployment (no API key needed)
client = QdrantClient(url=QDRANT_URL)

Now we setup our collection. With Qdrant we define the vector size and the distance metric used to compare vectors.

In [6]:
collection_name = "handbook-10-multiquery"

Creating a collection, we set `size` equal to the dimensionality of `gemini-embedding-001` (`3072`), and use `COSINE` distance which is compatible with it.

In [7]:
from qdrant_client.models import Distance, VectorParams

# create the collection if it does not already exist
if not client.collection_exists(collection_name):
    client.create_collection(
        collection_name=collection_name,
        vectors_config=VectorParams(size=3072, distance=Distance.COSINE),  # dimensionality of gemini-embedding-001
    )

# view collection info
client.get_collection(collection_name)

CollectionInfo(status=<CollectionStatus.GREEN: 'green'>, optimizer_status=<OptimizersStatusOneOf.OK: 'ok'>, warnings=None, indexed_vectors_count=0, points_count=0, segments_count=2, config=CollectionConfig(params=CollectionParams(vectors=VectorParams(size=3072, distance=<Distance.COSINE: 'Cosine'>, hnsw_config=None, quantization_config=None, on_disk=None, datatype=None, multivector_config=None), shard_number=1, sharding_method=None, replication_factor=1, write_consistency_factor=1, read_fan_out_factor=None, read_fan_out_delay_ms=None, on_disk_payload=True, sparse_vectors=None), hnsw_config=HnswConfig(m=16, ef_construct=100, full_scan_threshold=10000, max_indexing_threads=0, on_disk=False, payload_m=None, inline_storage=None), optimizer_config=OptimizersConfig(deleted_threshold=0.2, vacuum_min_vector_number=1000, default_segment_number=0, max_segment_size=None, memmap_threshold=None, indexing_threshold=10000, flush_interval_sec=5, max_optimization_threads=None, prevent_unoptimized=None)

Populate our index:

In [8]:
len(docs)

41584

In [9]:
# Cap the dataset so embedding stays within the free tier.
# We keep a small, configurable sample AND make sure the Llama 2 paper chunks
# are included, so the RAG answers below are actually grounded in real context.
SAMPLE_SIZE = 100  # increase for more data

llama2_docs = [d for d in docs if d.metadata["id"] == "2307.09288"]
other_docs = [d for d in docs if d.metadata["id"] != "2307.09288"]
docs = llama2_docs + other_docs[:SAMPLE_SIZE]
len(docs)

420

In [10]:
from langchain_qdrant import QdrantVectorStore

text_field = "text"

# wrap the Qdrant collection as a LangChain vector store
vectorstore = QdrantVectorStore(
    client=client,
    collection_name=collection_name,
    embedding=embed,
)

from tqdm.auto import tqdm

batch_size = 100

for i in tqdm(range(0, len(docs), batch_size)):
    i_end = min(len(docs), i+batch_size)
    docs_batch = docs[i:i_end]
    # add documents (Qdrant embeds the page_content and stores the metadata)
    vectorstore.add_documents(documents=docs_batch)

  0%|          | 0/5 [00:00<?, ?it/s]

## Multi-Query with LangChain

Now we switch across to using our populated collection as a vectorstore in Langchain.

In [11]:
from langchain_qdrant import QdrantVectorStore

text_field = "text"

# (re)connect to the existing Qdrant collection as a LangChain vector store
vectorstore = QdrantVectorStore(
    client=client,
    collection_name=collection_name,
    embedding=embed,
)

In [12]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    temperature=0,
    google_api_key=GOOGLE_API_KEY,
    model="gemini-2.5-flash",
)

# --- OpenRouter alternative (for reference) ---
# OpenRouter is OpenAI-compatible: use the ChatOpenAI class with a different base_url.
# Pick any ":free" model from https://openrouter.ai/models, e.g. "openai/gpt-oss-120b:free".
# from langchain_openai import ChatOpenAI
# llm = ChatOpenAI(
#     model="openai/gpt-oss-120b:free",
#     temperature=0,
#     api_key=OPENROUTER_API_KEY,
#     base_url="https://openrouter.ai/api/v1",
# )

We initialize the `MultiQueryRetriever`:

In [13]:
from langchain.retrievers.multi_query import MultiQueryRetriever

retriever = MultiQueryRetriever.from_llm(
    retriever=vectorstore.as_retriever(), llm=llm
)

We set logging so that we can see the queries as they're generated by our LLM.

In [14]:
# Set logging for the queries
import logging

logging.basicConfig()
logging.getLogger("langchain.retrievers.multi_query").setLevel(logging.INFO)

To query with our multi-query retriever we call the `get_relevant_documents` method.

In [15]:
question = "tell me about llama 2?"

docs = retriever.invoke(question)
len(docs)

INFO:langchain.retrievers.multi_query:Generated queries: ['What is Llama 2 and what are its primary capabilities?', 'Can you provide details on the architecture, training, and performance of Llama 2?', 'What are the key differences between Llama 2 and other large language models, and what are its typical use cases?']


7

From this we get a variety of docs retrieved by each of our queries independently. By default the `retriever` is returning `3` docs for each query — totalling `9` documents — however, as there is some overlap we actually return `6` unique docs.

In [16]:
docs

[Document(metadata={'title': 'Llama 2: Open Foundation and Fine-Tuned Chat Models', 'source': 'http://arxiv.org/pdf/2307.09288', 'id': '2307.09288', 'chunk-id': '9', 'text': 'asChatGPT,BARD,andClaude. TheseclosedproductLLMsareheavilyﬁne-tunedtoalignwithhuman\npreferences, which greatly enhances their usability and safety. This step can require signiﬁcant costs in\ncomputeandhumanannotation,andisoftennottransparentoreasilyreproducible,limitingprogresswithin\nthe community to advance AI alignment research.\nIn this work, we develop and release Llama 2, a family of pretrained and ﬁne-tuned LLMs, L/l.sc/a.sc/m.sc/a.sc /two.taboldstyle and\nL/l.sc/a.sc/m.sc/a.sc /two.taboldstyle-C/h.sc/a.sc/t.sc , at scales up to 70B parameters. On the series of helpfulness and safety benchmarks we tested,\nL/l.sc/a.sc/m.sc/a.sc /two.taboldstyle-C/h.sc/a.sc/t.sc models generally perform better than existing open-source models. They also appear to\nbe on par with some of the closed-source models, at least on

## Adding the Generation in RAG

So far we've built a multi-query powered **R**etrieval **A**ugmentation chain. Now, we need to add **G**eneration.

In [17]:
from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain

QA_PROMPT = PromptTemplate(
    input_variables=["query", "contexts"],
    template="""You are a helpful assistant who answers user queries using the
    contexts provided. If the question cannot be answered using the information
    provided say "I don't know".

    Contexts:
    {contexts}

    Question: {query}""",
)

# Chain
qa_chain = LLMChain(llm=llm, prompt=QA_PROMPT)

C:\Users\pc\AppData\Local\Temp\ipykernel_48192\541099612.py:17: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 1.0. Use :meth:`~RunnableSequence, e.g., `prompt | llm`` instead.
  qa_chain = LLMChain(llm=llm, prompt=QA_PROMPT)


In [18]:
out = qa_chain.invoke(
    {
        "query": question,
        "contexts": (chr(10)+"---"+chr(10)).join([d.page_content for d in docs])
    }
)
out["text"]

"Llama 2 is a family of pretrained and fine-tuned large language models (LLMs) developed and released by Meta's GenAI.\n\nHere are some key details about Llama 2:\n\n*   **Scale:** It ranges in scale from 7 billion to 70 billion parameters.\n*   **Versions:** It includes pretrained models (Llama 2) and fine-tuned models optimized for dialogue use cases (Llama 2-Chat).\n*   **Performance:**\n    *   Llama 2-Chat models generally perform better than existing open-source models on helpfulness and safety benchmarks.\n    *   They also appear to be on par with some closed-source models like ChatGPT, Bard, and Claude, based on human evaluations.\n*   **Training:**\n    *   Trained on 2 trillion tokens of data from publicly available sources.\n    *   The training corpus does not include data from Meta's products or services, and efforts were made to remove data from sites known to contain a high volume of personal information.\n    *   It features a doubled context length and uses grouped-qu

## Chaining Everything with a SequentialChain

We can pull together the logic above into a function or set of methods, whatever is prefered — however if we'd like to use LangChain's approach to this we must "chain" together multiple chains. The first retrieval component is (1) not a chain per se, and (2) requires processing of the output. To do that, and fit with LangChain's "chaining chains" approach, we setup the _retrieval_ component within a `TransformChain`:

In [19]:
from langchain.chains import TransformChain

def retrieval_transform(inputs: dict) -> dict:
    docs = retriever.invoke(inputs["question"])
    docs = [d.page_content for d in docs]
    docs_dict = {
        "query": inputs["question"],
        "contexts": (chr(10)+"---"+chr(10)).join(docs)
    }
    return docs_dict

retrieval_chain = TransformChain(
    input_variables=["question"],
    output_variables=["query", "contexts"],
    transform=retrieval_transform
)

Now we chain this with our generation step using the `SequentialChain`:

In [20]:
from langchain.chains import SequentialChain

rag_chain = SequentialChain(
    chains=[retrieval_chain, qa_chain],
    input_variables=["question"],  # we need to name differently to output "query"
    output_variables=["query", "contexts", "text"]
)

Then we perform the full RAG pipeline:

In [21]:
out = rag_chain.invoke({"question": question})
out["text"]

INFO:langchain.retrievers.multi_query:Generated queries: ['What is Llama 2 and what are its primary capabilities?', 'Can you provide details on the architecture, training, and performance of Llama 2?', 'What are the key differences between Llama 2 and other large language models, and what are its typical use cases?']


"Llama 2 is a family of pretrained and fine-tuned large language models (LLMs) developed and released by Meta's GenAI.\n\nHere are some key details about Llama 2:\n\n*   **Scale:** It ranges in scale from 7 billion to 70 billion parameters.\n*   **Versions:** It includes pretrained models (Llama 2) and fine-tuned models optimized for dialogue use cases (Llama 2-Chat).\n*   **Performance:** Llama 2-Chat models generally perform better than existing open-source models on helpfulness and safety benchmarks. They also appear to be on par with some closed-source models like ChatGPT, Bard, and Claude, based on human evaluations.\n*   **Training Data:** It was trained on 2 trillion tokens of data from publicly available sources. Efforts were made to remove data from sites known to contain a high volume of personal information, and it does not include data from Meta's products or services.\n*   **Improvements over Llama 1:** It uses more tokens for training, has a doubled context length, and in

---

## Custom Multiquery

We'll try this with two prompts, both encourage more variety in search queries.

**Prompt A**
```
Your task is to generate 3 different search queries that aim to
answer the user question from multiple perspectives.
Each query MUST tackle the question from a different viewpoint,
we want to get a variety of RELEVANT search results.
Provide these alternative questions separated by newlines.
Original question: {question}
```


**Prompt B**
```
Your task is to generate 3 different search queries that aim to
answer the user question from multiple perspectives. The user questions
are focused on Large Language Models, Machine Learning, and related
disciplines.
Each query MUST tackle the question from a different viewpoint, we
want to get a variety of RELEVANT search results.
Provide these alternative questions separated by newlines.
Original question: {question}
```

In [22]:
from typing import List
from langchain.chains import LLMChain
from langchain.prompts import PromptTemplate
from langchain_core.output_parsers import BaseOutputParser


# Output parser splits the LLM result into a list of query strings.
# In langchain 0.3.x the MultiQueryRetriever consumes this list directly,
# so the parser returns List[str] (one query per line).
class LineListOutputParser(BaseOutputParser[List[str]]):
    def parse(self, text: str) -> List[str]:
        lines = text.strip().split(chr(10))
        return [ln.strip() for ln in lines if ln.strip()]


output_parser = LineListOutputParser()

template = """
Your task is to generate 3 different search queries that aim to
answer the user question from multiple perspectives. The user questions
are focused on Large Language Models, Machine Learning, and related
disciplines.
Each query MUST tackle the question from a different viewpoint, we
want to get a variety of RELEVANT search results.
Provide these alternative questions separated by newlines.
Original question: {question}
"""

QUERY_PROMPT = PromptTemplate(
    input_variables=["question"],
    template=template,
)
llm = ChatGoogleGenerativeAI(
    temperature=0,
    google_api_key=GOOGLE_API_KEY,
    model="gemini-2.5-flash",
)

# Chain
llm_chain = LLMChain(llm=llm, prompt=QUERY_PROMPT, output_parser=output_parser)

In [23]:
# Run
# In langchain 0.3.x the MultiQueryRetriever reads the llm_chain output (our
# List[str] of queries) directly, so we no longer pass the legacy parser_key arg.
from langchain.retrievers.multi_query import MultiQueryRetriever

retriever = MultiQueryRetriever(
    retriever=vectorstore.as_retriever(),
    llm_chain=llm_chain,
)

# Results
docs = retriever.invoke(question)
len(docs)

INFO:langchain.retrievers.multi_query:Generated queries: ['Llama 2 model explained', 'Llama 2 architecture and capabilities', 'Llama 2 applications and comparisons']


7

In [24]:
docs

[Document(metadata={'title': 'Llama 2: Open Foundation and Fine-Tuned Chat Models', 'source': 'http://arxiv.org/pdf/2307.09288', 'id': '2307.09288', 'chunk-id': '9', 'text': 'asChatGPT,BARD,andClaude. TheseclosedproductLLMsareheavilyﬁne-tunedtoalignwithhuman\npreferences, which greatly enhances their usability and safety. This step can require signiﬁcant costs in\ncomputeandhumanannotation,andisoftennottransparentoreasilyreproducible,limitingprogresswithin\nthe community to advance AI alignment research.\nIn this work, we develop and release Llama 2, a family of pretrained and ﬁne-tuned LLMs, L/l.sc/a.sc/m.sc/a.sc /two.taboldstyle and\nL/l.sc/a.sc/m.sc/a.sc /two.taboldstyle-C/h.sc/a.sc/t.sc , at scales up to 70B parameters. On the series of helpfulness and safety benchmarks we tested,\nL/l.sc/a.sc/m.sc/a.sc /two.taboldstyle-C/h.sc/a.sc/t.sc models generally perform better than existing open-source models. They also appear to\nbe on par with some of the closed-source models, at least on

Putting this together in another `SequentialChain`:

In [25]:
retrieval_chain = TransformChain(
    input_variables=["question"],
    output_variables=["query", "contexts"],
    transform=retrieval_transform
)

rag_chain = SequentialChain(
    chains=[retrieval_chain, qa_chain],
    input_variables=["question"],  # we need to name differently to output "query"
    output_variables=["query", "contexts", "text"]
)

And asking again:

In [26]:
out = rag_chain.invoke({"question": question})
out["text"]

INFO:langchain.retrievers.multi_query:Generated queries: ['Llama 2 explained', 'Llama 2 architecture and training data', 'Llama 2 capabilities and licensing']


"Llama 2 is a family of pretrained and fine-tuned large language models (LLMs) developed and released by Meta's GenAI.\n\nHere are some key details about Llama 2:\n*   **Scale:** It ranges in size from 7 billion to 70 billion parameters.\n*   **Versions:** The family includes Llama 2 (pretrained models) and Llama 2-Chat (fine-tuned models optimized for dialogue use cases).\n*   **Performance:** Llama 2-Chat models generally perform better than existing open-source models on helpfulness and safety benchmarks. They also appear to be on par with some closed-source models based on human evaluations.\n*   **Training:**\n    *   Trained on 2 trillion tokens of publicly available data, which does not include data from Meta's products or services. Efforts were made to remove data from sites known to contain a high volume of personal information.\n    *   It was trained on 40% more total tokens than Llama 1, doubled the context length, and used Grouped-Query Attention (GQA) for improved inferen

After finishing, delete your Qdrant collection to save resources:

In [27]:
client.delete_collection(collection_name)

True

---